# v.in.ags - Importing ArcGIS Server Data into GRASS

This notebook demonstrates how to use the **v.in.ags** GRASS addon to download
vector features directly from an ArcGIS Server (AGS) REST API and import them
as GRASS vector maps.

**What you will learn:**

1. Start a GRASS session inside Jupyter
2. List layers available in an AGS feature service
3. Perform a basic import of all features
4. Filter features by attribute (SQL WHERE clause)
5. Filter features by bounding box (bbox_filter)
6. Import only selected attribute fields
7. Import into a projected project (reprojection is automatic)
8. Fix polygon topology with snapping
9. Inspect and visualise the imported map


## 1. Start a GRASS session

We create a temporary GRASS project in the WGS84 geographic coordinate system
(EPSG:4326). For production workflows you would use your own project.

In [ ]:
import subprocess
import sys

# check where GRASS Python packages are and add them to path
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

In [ ]:
import tempfile
import os
from pathlib import Path

# grass.jupyter provides a convenient session manager
import grass.jupyter as gj
import grass.script as gs
from grass.tools import Tools

# Create a temporary GRASS project (WGS84) for this tutorial
tempdir = tempfile.TemporaryDirectory()
project_path = Path(tempdir.name, "wgs84_project")

gs.create_project(path=tempdir.name, name="wgs84_project", epsg="4326")
# Start GRASS in the recently created project
session = gj.init(project_path)
print("GRASS session started")
tools = Tools(session=session)

## 2. Install the addon (if not already installed)

In [ ]:
# Install v.in.ags from the GRASS addons repository.
# Skip this cell if you have already installed the addon.
gs.run_command("g.extension", extension="v.in.ags")
# Install if running from the grass-addons
# gs.run_command("g.extension", extension="v.in.ags", url=str(Path().parent.cwd().as_uri()))

## 3. List available layers in a service

Before importing, use the **-l** flag to explore which layers a service
exposes. We use Esri's public sample server here.

In [ ]:
SERVICE_ROOT = (
    "https://sampleserver6.arcgisonline.com/arcgis/rest/services/USA/MapServer"
)

print(tools.v_in_ags(flags="l", url=SERVICE_ROOT).stdout)

The output lists each layer's numeric **ID**, its **type**, and its **name**.
Use the ID with the `layer` parameter when importing from a service root URL.

## 4. Basic import – all features from a layer

Import all features from layer 0 (cities) of the sample MapServer.
No reprojection is applied; the output map will be in WGS84.

In [ ]:
# Hardcode layer ID
CITIES_URL = SERVICE_ROOT + "/0"  # layer 0 = cities

#  or Dynamically grab layer ID
# layers_data = tools.v_in_ags(flags="l", url=SERVICE_ROOT, format="json").json
# CITIES_LAYER_ID = list(filter(lambda i: "Cities" == i.get('name'), layers_data))[0].get('id')
# CITIES_URL = SERVICE_ROOT + "/" + str(CITIES_LAYER_ID)

tools.v_in_ags(
    url=CITIES_URL,
    output="usa_cities",
    overwrite=True,
)

# Confirm import
info = gs.vector_info_topo("usa_cities")
print("Points imported:", info["points"])

## 5. Attribute filter – SQL WHERE clause

Import only cities in California. The **where** parameter accepts any SQL
expression supported by the server.

In [ ]:
tools.v_in_ags(
    url=CITIES_URL,
    output="ca_cities",
    where="st = 'CA'",
    overwrite=True,
)

info = tools.v_info(map="ca_cities", format="json").json
print("California cities imported:", info["points"])

# Another way to get the vector info
# info = gs.vector_info_topo("ca_cities")
# print("California cities imported:", info["points"])

## 6. Spatial filter - bounding box

The **bbox_filter** option restricts the download to features that intersect a
bounding box on the server. Coordinates must be in WGS84 (decimal degrees):
`xmin,ymin,xmax,ymax`. (Use **extent=region** instead to filter by the current
computational region.)

In [ ]:
# Bounding box covering the US Pacific Northwest
PNW_EXTENT = "-125,42,-116,49"

tools.v_in_ags(
    url=CITIES_URL,
    output="pnw_cities",
    bbox_filter=PNW_EXTENT,
    overwrite=True,
)

info = tools.v_info(map="pnw_cities", format="json").json
print("Pacific Northwest cities imported:", info["points"])

## 7. Combine attribute and spatial filters

In [ ]:
# Large cities (population > 100 000) anywhere in the continental US
CONUS_EXTENT = "-125,24,-66,50"

tools.v_in_ags(
    url=CITIES_URL,
    output="large_cities",
    where="pop2000 > 100000",
    bbox_filter=CONUS_EXTENT,
    overwrite=True,
)

info = gs.vector_info_topo("large_cities")
print("Large cities imported:", info["points"])

## 8. Selective field import

Use **fields** to download only a subset of attributes. This reduces
transfer size for wide tables.

In [ ]:
tools.v_in_ags(
    url=CITIES_URL,
    output="cities_slim",
    fields="areaname,st,pop2000",
    overwrite=True,
)

# Show column names
columns = tools.v_info(map="cities_slim", format="json", flags="c").json
print(columns)

## 9. Import into a projected project (automatic reprojection)

*v.in.ags* always imports through *v.import*, which reprojects the downloaded
data into the current project CRS automatically (and imports directly when the
project is already WGS84). No flag is needed.

Here we demonstrate by creating a UTM project and importing into it.

In [ ]:
# Create a second project in UTM Zone 10N (EPSG:32610)
utm_path = Path(tempdir.name, "utm10n_project")
gs.create_project(path=tempdir.name, name="utm10n_project", epsg="32610")
utm_session = gj.init(utm_path)
tools = Tools(session=utm_session)

tools.v_in_ags(
    url=CITIES_URL,
    output="pnw_cities_utm",
    bbox_filter=PNW_EXTENT,
    overwrite=True,
)

# The output map is in the project CRS (UTM Zone 10N)
print(tools.g_proj(format="plain", flags="p").stdout)
print("Points:", gs.vector_info_topo("pnw_cities_utm")["points"])

## 10. Visualise the imported data

Switch back to the WGS84 project and use `grass.jupyter.Map` to display
the imported cities layer.

In [ ]:
# Return to the WGS84 session
session = gj.init(project_path)
tools = Tools(session=session)
# Set the region to match the large_cities extent
tools.g_region(vector="large_cities", grow=2)

# Display with grass.jupyter
m = gj.Map()
m.d_vect(
    map="large_cities", icon="basic/circle", size=8, color="blue", fill_color="cyan"
)
m.d_grid(size=5, color="grey")
m.show()

## 11. Post-import analysis

Once data is in GRASS, the full suite of GRASS vector tools is available.

In [ ]:
# Basic attribute statistics on imported population field
stats = tools.v_univar(
    map="large_cities", column="pop2000", type="point", format="json"
)

print("Count :", stats["n"])
print("Min   :", stats["min"])
print("Max   :", stats["max"])
print("Mean  :", stats["mean"])

In [ ]:
# Query the top 5 cities by population
top5 = tools.v_db_select(
    map="large_cities",
    columns="areaname,st,pop2000",
    where="pop2000 > 500000",
    format="csv",
).stdout
print(top5)

## 12. Polygon data and snapping

Polygon layers from ArcGIS Server frequently have invalid topology and may
import with no areas. The **snap** option snaps boundary vertices so areas
build. Below we import North Carolina counties (polygons) from the token-free
sample server.

In [ ]:
# Import county boundaries for California (polygons)
COUNTIES_URL = (
    "https://sampleserver6.arcgisonline.com/arcgis/rest/services/USA/MapServer/3"
)

tools.v_in_ags(
    url=COUNTIES_URL,
    output="nc_counties",
    where="state_name = 'North Carolina'",
    fields="name,state_name,pop2000",
    snap=1e-6,
    overwrite=True,
)

tools.v_in_ags(
    url=CITIES_URL,
    output="nc_large_cities",
    where="pop2000 > 50000 AND st = 'NC'",
    overwrite=True,
)

# Show the result
info = gs.vector_info_topo("nc_counties")
print("North Carolina counties imported:", info["areas"])
info = gs.vector_info_topo("nc_large_cities")
print("North Carolina cities > 50,000 imported:", info["points"])

## 13. Visualise the imported data polygon data

Use `grass.jupyter.Map` to display the imported NC cities (`nc_large_cities`) layer with the NC counties (`nc_counties`).

In [ ]:
# Display with grass.jupyter
with gs.RegionManager(vector="nc_counties"):
    m = gj.Map(use_region=True)
    m.d_vect(map="nc_counties", size=8, color="yellow", fill_color="#737000")
    m.d_vect(
        map="nc_large_cities",
        icon="basic/circle",
        size=8,
        color="blue",
        fill_color="cyan",
    )

    m.d_grid(size="0", color="grey", flags="a")
    m.show()

## 13. Clean up

In [ ]:
# Remove the temporary GRASS projects
tempdir.cleanup()
print("Temporary projects removed.")

## Summary

| Task | Command |
|------|---------|
| List layers | `v.in.ags -l url=<service_root>` |
| Import all features | `v.in.ags url=<layer_url> output=<name>` |
| Attribute filter | `... where="field = 'value'"` |
| Bounding box filter | `... bbox_filter="xmin,ymin,xmax,ymax"` |
| Region filter | `... extent=region` |
| Selective fields | `... fields="col1,col2"` |
| Fix polygon topology | `... snap=1e-6` |
| Request a CRS from server | `... outsr=<WKID>` |
| Reproject to project CRS | automatic (via *v.import*) |

For full documentation see `v.in.ags --help` or the
[GRASS addons manual](https://grass.osgeo.org/grass-devel/manuals/addons/v.in.ags.html).